Imports

In [93]:
import pandas as pd
import re
import ast

In [94]:
oscars = pd.read_csv("the_oscar_award.csv")



In [95]:
oscars_film = (
    oscars
    .groupby(["film", "year_film"])
    .agg(
        oscar_nominations=("category","count"),
        oscar_wins = ("winner", "sum")
    )
    .reset_index()
)

In [96]:
oscars_film["oscar_nominated"] = oscars_film["oscar_nominations"] > 0
oscars_film["oscar_won"] = oscars_film["oscar_wins"] > 0

In [97]:
def clean_title(title):

    title = str(title).lower()
    title = re.sub(r"\(.*?\)", "", title)
    title = re.sub(r"[^a-z0-9 ]", "", title)
    title = title.strip()

    return title

In [98]:
df_full = pd.read_csv("movies_dataset.csv")
df_full["title_clean"] = df_full["title"].apply(clean_title)
oscars_film["title_clean"] = oscars_film["film"].apply(clean_title)

In [99]:
df_full["release_year"] = pd.to_datetime(df_full["release_date"]).dt.year

In [100]:
df_merged = df_full.merge(
    oscars_film,
    left_on=["title_clean", "release_year"],
    right_on=["title_clean", "year_film"],
    how="left"
)

In [101]:
df_merged["oscar_nominations"] = df_merged["oscar_nominations"].fillna(0)
df_merged["oscar_wins"] = df_merged["oscar_wins"].fillna(0)

df_merged["oscar_nominated"] = df_merged["oscar_nominated"].fillna(False)
df_merged["oscar_won"] = df_merged["oscar_won"].fillna(False)

In [102]:
def extract_main_genre(x):
    
    if pd.isna(x):
        return None
    
    try:
        genres = ast.literal_eval(x)
        
        if isinstance(genres, list) and len(genres) > 0:
            return genres[0]
    except:
        return None

In [103]:
df_merged["main_genre"] = df_merged["genres"].apply(extract_main_genre)

In [104]:
df_merged["oscar_nominated"].mean()

np.float64(0.0413447782546495)

In [105]:
keep_columns = [
    "id",
    "title",
    "release_year",
    "budget",
    "revenue",
    "runtime",
    "main_genre",
    "popularity",
    "oscar_nominations",
    "oscar_wins",
    "oscar_nominated",
    "oscar_won"
]

In [106]:
df_merged = df_merged[keep_columns].copy()

In [107]:
df_merged = df_merged[
    (df_merged["budget"] > 0) &
    (df_merged["revenue"] > 0)
]

In [108]:
df_merged.to_csv("oscars_movies_merged.csv", index=False)


In [109]:
df_merged.head()

,id,title,release_year,budget,revenue,runtime,main_genre,popularity,oscar_nominations,oscar_wins,oscar_nominated,oscar_won
2,769,GoodFellas,1990,25000000.0,47072327.0,145.0,Drama,19.8633,0.0,0.0,False,False
13,530,A Grand Day Out,1990,17300.0,40118.0,24.0,Family,18.4738,1.0,0.0,True,False
20,1573,Die Hard 2,1990,70000000.0,240031094.0,124.0,Action,12.1105,0.0,0.0,False,False
21,162,Edward Scissorhands,1990,20000000.0,86024005.0,105.0,Fantasy,14.4765,1.0,0.0,True,False
23,196,Back to the Future Part III,1990,40000000.0,244527583.0,119.0,Adventure,11.6486,0.0,0.0,False,False


In [112]:
df_analysis["oscar_won"].sum()

216